In [1]:
%matplotlib widget
# Boilerplate import code for all libraries
# Changes to the precision require re-loading the kernel and need to be done before any op uses them.
import warpSPHCore_config as swc
from typing import Any
swc.configure(precision="float32", dim=Any) # precision: float16|half|float32|single|float64|double

import warpSPHCore as sph
from warpSPHCore.type_config import *
print(get_type_config()) # confirms active settings

# Initialize warp at this point
import warp as wp; wp.init()

import os
import torch
if torch.cuda.is_available(): # set the TORCH_CUDA_ARCH_LIST environment variable to the compute capability of the GPU for faster compiles
    os.environ['TORCH_CUDA_ARCH_LIST'] = f'{torch.cuda.get_device_properties(0).major}.{torch.cuda.get_device_properties(0).minor}'

import warnings
from tqdm import TqdmExperimentalWarning
warnings.filterwarnings("ignore", category=TqdmExperimentalWarning)
from tqdm.autonotebook import tqdm

# final import blocks that are generic
import matplotlib.pyplot as plt
from torch.profiler import profile, record_function, ProfilerActivity
import numpy as np
import math
import shlex    
import subprocess
import shutil

# custom SPH libraries
from warpSPHIntegrators.integration import *
from warpSPHCore import *
from warpSPHPlotting import *

# This library
from warpSPH import *

# The case utilities that contain all the case setup functions for the various test cases
from warpSPH.caseUtils import *

{'scalar_t': <class 'warp._src.types.float32'>, 'dim_t': typing.Any}
Warp 1.12.0 initialized:
   CUDA Toolkit 12.9, Driver 13.2
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "NVIDIA RTX PRO 500 Blackwell Generation Laptop GPU" (6 GiB, sm_120, mempool enabled)
   Kernel cache:
     /home/lu26029/.cache/warp/1.12.0


In [2]:
directory = f'./compressed/'

simulations = os.listdir(directory)
simulations = [sim for sim in simulations if 'h5' in sim]


print(f'Found {len(simulations)} simulations in {directory}')
# trajectoryFile = f'{directory}/trajectory.h5'
# configFile = f'{directory}/config.json'

Found 1 simulations in ./compressed/


In [3]:
import h5py
import json

file = simulations[0]
trajectoryFile = f'{directory}/{file}'

# loadedConfig = json.load(open(configFile, 'r'))

trajectory = h5py.File(trajectoryFile, 'r')
numStates = trajectory['positions'].shape[0]
numFluidParticles = trajectory['positions'].shape[1]
print(f'Loaded trajectory with {numStates} states and {numFluidParticles} fluid particles from {trajectoryFile}')

Loaded trajectory with 20 states and 60690 fluid particles from ./compressed//trajectory_openChannel_2026-07-30_11-46-08_256_4_2.0_6.0_obstacle_0.25_0_-1.5.h5


In [4]:
def restoreConfig_from_h5(group, indent=0):
    config = {}
    # print('  '*indent + f'Restoring config from h5 group: {group.name}, keys: {list(group.keys())}, attributes: {list(group.attrs.keys())}')
    for key, value in group.attrs.items():
        if key != 'taggedType':
            # print('  '*indent + f'Restoring attribute for key: {key}, value: {value}')
            config[key] = value
    for key, subgroup in group.items():
        if subgroup.attrs['taggedType'] == 'dict':
            # print('  '*indent + f'Restoring dict for key: {key}, keys: {list(subgroup.keys())}')
            config[key] = restoreConfig_from_h5(subgroup, indent + 1)
        elif subgroup.attrs['taggedType'] == 'list':
            # print(f'Restoring list for key: {key}, length: {len(subgroup)}')
            config[key] = [restoreConfig_from_h5(subgroup[f'item_{i}'], indent + 1) for i in range(len(subgroup))]
            subkeys = list(subgroup.attrs.keys())
            subkeys = [k for k in subkeys if k.startswith('item_')]
            # print(f'Found {len(subkeys)} items in subgroup {key}: {subkeys}')
            for i in range(len(subkeys)):
                item_key = f'item_{i}'
                if item_key in subgroup.attrs:
                    # print(f'Restoring item_{i} for list key: {key}, value: {subgroup.attrs[item_key]}')
                    config[key].append(subgroup.attrs[item_key])
                # else:
                    # print(f'Warning: item_{i} not found in subgroup {key}')
            # print('  '*indent + f'Restored list for key: {key}, length: {len(config[key])} [{config[key]}]')
        else:
            print('  '*indent + f'Warning: Unknown type for subgroup {key}: {subgroup.attrs["taggedType"]}')
            raise ValueError(f'Unknown type for subgroup {key}: {subgroup.attrs["taggedType"]}')
    return config

restoredConfig = restoreConfig_from_h5(trajectory['config'])

In [5]:
display(restoredConfig)

{'scheme': 'deltaSPH',
 'timestamp': '2026-07-30_11-46-10',
 'config': {'adaptiveDt': np.True_,
  'cflFactor': np.float64(0.3),
  'device': 'cuda:0',
  'dim': np.int64(2),
  'dt': np.float64(0.0002),
  'dtGrowthFactor': np.float64(1.1),
  'dtype': 'torch.float32',
  'gradientMode': 'Difference',
  'integrationScheme': 'rungeKutta2',
  'kernel': 'Wendland4',
  'laplacianMode': 'Brookshaw',
  'maxDt': np.float64(0.01),
  'minDt': np.float64(1e-08),
  'samplingScheme': 'regular',
  'supportMode': 'KernelMeanSymmetric',
  'targetNeighbors': np.float64(50.26548245743669),
  'verletScale': np.float64(1.4142135623730951),
  'domain': {'dim': np.int64(2),
   'max': [np.float64(3.0), np.float64(1.0390625)],
   'min': [np.float64(-3.0), np.float64(-1.0390625)],
   'periodic': [np.True_, np.True_]}},
 'schemeConfig': {'adaptiveSupportCorrections': np.True_,
  'adaptiveSupportIterations': np.int64(1),
  'adaptiveSupportScheme': 'NoScheme',
  'adaptiveSupportThreshold': np.float64(0.001),
  'bandwi

In [6]:
from warpSPH.io import *


scheme = restoredConfig['scheme']
schemeEnum = schemeNameToSimulationScheme(scheme)

SimulationSystem, SimulationState, SimulationConfig, SimulationUpdate, fn, export_fn, import_fn = buildScheme(schemeEnum)


config, schemeConfig = dictToConfig(restoredConfig['config']), import_fn(restoredConfig['schemeConfig'])

In [7]:
schemeConfig.regions

[ParticleRegion(sdf=<function build_sdfs.<locals>.<lambda> at 0x7fcff82b7880>, type=<RegionType.Boundary: 2>, particles=None, contour=None, initialConditions={}, kind=<BCType.constant: 1>),
 ParticleRegion(sdf=<function <lambda> at 0x7fd3735d7ec0>, type=<RegionType.Fluid: 1>, particles=None, contour=None, initialConditions={}, kind=<BCType.zeros: 5>)]

In [8]:
device = torch.device(config.device)
kinds = torch.tensor(trajectory['combinedKinds'][:], dtype=torch.int32)

simulationState = WeaklyCompressibleState(
    positions = torch.tensor(trajectory['combinedPositions'][:], dtype=torch.float32, device = device)[kinds!=2],
    supports = torch.tensor(trajectory['combinedSupports'][:], dtype=torch.float32, device = device)[kinds!=2],
    masses = torch.tensor(trajectory['combinedMasses'][:], dtype=torch.float32, device = device)[kinds!=2],
    densities = torch.tensor(trajectory['combinedDensities'][:], dtype=torch.float32, device = device)[kinds!=2],
    velocities = torch.tensor(trajectory['combinedVelocities'][:], dtype=torch.float32, device = device)[kinds!=2],

    pressures = None,
    soundspeeds = None,

    kinds = torch.tensor(trajectory['combinedKinds'][:], dtype=torch.int32, device = device)[kinds!=2],
    materials = torch.tensor(trajectory['combinedMaterials'][:], dtype=torch.int32, device = device)[kinds!=2],
    UIDs = torch.tensor(trajectory['combinedUIDs'][:], dtype=torch.int64, device = device)[kinds!=2],
    UIDcounter = int(trajectory['combinedPositions'][:][kinds!=2].shape[0]),

    ghostIndices = torch.tensor(trajectory['combinedGhostIndices'][:], dtype=torch.int32, device = device)[kinds!=2],
    ghostOffsets = torch.tensor(trajectory['combinedGhostOffsets'][:], dtype=torch.float32, device = device)[kinds!=2]
)

particleState = addBoundaryGhostParticles(schemeConfig.regions, simulationState)


rigidBodyIDs = torch.unique(particleState.materials[particleState.kinds == 1]).cpu().numpy()
# print(rigidBodyIDs)
rigidBodies = []
for id in rigidBodyIDs:
    # print('Processing rigid body', id)
    rigidBody = buildRigidBody(particleState, schemeConfig.regions, id)
    if(rigidBody is not None):
        rigidBodies.append(rigidBody)

for rigidBody in rigidBodies:
    particleState = updateBodyParticlesWCSPH(particleState, rigidBody)

schemeConfig.rigidBodies = rigidBodies
# schemeConfig.regions = 
config.regions = schemeConfig.regions

compressibleSystem = SimulationSystem(
    state=particleState, 
    adjacency = None, 
    domain = config.domain
)

In [16]:
nx = int(trajectory.attrs['nx'])
markerSize = float(trajectory.attrs['markerSize'])
plotWidth = float(trajectory.attrs['plotWidth'])
plotHeight = float(trajectory.attrs['plotHeight'])
plotDensity = bool(trajectory.attrs['plotDensity'])
n_h = float(trajectory.attrs['n_h'])


L = float(trajectory.attrs['L'])
W = float(trajectory.attrs['W'])

timeLimit = float(trajectory.attrs['timeLimit'])
enableFreestream = bool(trajectory.attrs['enableFreestream'])
forcingWidth = float(trajectory.attrs['forcingWidth'])
freeStreamVelocity = float(trajectory.attrs['freeStreamVelocity'])
band = int(trajectory.attrs['band'])

targetDt = float(trajectory.attrs['targetDt'])

caseName = str(trajectory.attrs['caseName'])
plot = bool(trajectory.attrs['plot'])
plotInterval = int(trajectory.attrs['plotInterval'])

disableGravity = bool(trajectory.attrs['disableGravity'])
gravityDirection = np.array(trajectory.attrs['gravityDirection'][:]).tolist()
gravityMagnitude = float(trajectory.attrs['gravityMagnitude'])

enableSloshing = bool(trajectory.attrs['enableSloshing'])
sloshingAmplitude = float(trajectory.attrs['sloshingAmplitude'])
sloshingFrequency = float(trajectory.attrs['sloshingFrequency'])

obstacleActive = bool(trajectory.attrs['obstacleActive'])
obstacleType = str(trajectory.attrs['obstacleType'])

offsetX = float(trajectory.attrs['offsetX'])
aoa = float(trajectory.attrs['aoa'])
maxExtent = float(trajectory.attrs['maxExtent'])

fillRatio = float(trajectory.attrs['fillRatio'])
semiPeriodic = bool(trajectory.attrs['semiPeriodic'])
fullyPeriodic = bool(trajectory.attrs['fullyPeriodic'])
fluidWidth = float(trajectory.attrs['fluidWidth'])

enableNoise = bool(trajectory.attrs['enableNoise'])
octaves = int(trajectory.attrs['octaves'])
lacunarity = int(trajectory.attrs['lacunarity'])
persistence = float(trajectory.attrs['persistence'])
baseFrequency = int(trajectory.attrs['baseFrequency'])
kind = str(trajectory.attrs['kind'])
seed = int(trajectory.attrs['seed'])
noiseAmplitude = float(trajectory.attrs['noiseAmplitude'])
bandWidth = float(trajectory.attrs['bandWidth'])

enableKolmogorovForcing = bool(trajectory.attrs['enableKolmogorovForcing'])
kolmogorovForcingAmplitude = float(trajectory.attrs['kolmogorovForcingAmplitude'])
kolmogorovForcingWavenumber = int(trajectory.attrs['kolmogorovForcingWavenumber'])
exportInterval = int(trajectory.attrs['exportInterval'])

config.dx = L/nx

u_freestream = freeStreamVelocity

In [17]:
schemeConfig.fluid.fixedSoundSpeed

26.98631979252399

In [18]:
runningState = compressibleSystem.initializeNewState()

In [19]:

markerSize = 8
plotter = visualize(
    particleState = runningState.state,
    domain = config.domain,
    quantities = {
        "A": runningState.state.velocities,
        "B":runningState.state.UIDs
    },
    plotOptions = {
        "A": PlottingOptions(
            colorMap = UniformColorMap.viridis,
            markerSize = markerSize,
            midPoint = 0.0,
            quantityScaling = PlotScaling.Linear,
            mapping = Mapping.L2Norm,
            plotTitle = "velocities",
            plotTitleGap = 0.08,
            boundaryVisualization = VisualizeOptions.Passive,
            # gridVisualization = GridVisualization(
            #     resolution = 512,
            # ),
            # vMin=1e-10
            # vMin = 0.0,
            # vMax = schemeConfig.fluid.fixedSoundSpeed * 0.1
        ),
        "B": PlottingOptions(
            colorMap = CyclicColorMap.twilight,
            # flipColorMap=True,
            markerSize = markerSize,
            # midPoint = 1.0,
            quantityScaling = PlotScaling.Linear,
            plotTitle = "UIDs",
            # vMin = 0.95,
            # vMax = 1.05
            plotTitleGap = 0.08,
            # gridVisualization = GridVisualization(
            #     resolution = 512,
            # ),
        )
    },
    figTitle = "Taylor Green Vortex",
    mosaic = 'AB',
    figsize= (14,5),
    backend='vispy',
    # backend='pyVista',
    # backendOptions = {
    #     # In notebooks, use trame for reliable live updates.
    #     'jupyter_backend': 'trame',
    # }
)


RFBOutputContext()

In [22]:
integrator = getIntegrator(config.integrationScheme)
config.dim = 2

In [25]:
t_limit = 1.28
nSteps = int(t_limit / config.dt)

runningState = compressibleSystem.initializeNewState()
runningState.adjacency = None
# schemeConfig.rigidBodies[0].linearVelocity = 0.0

kes = []
priorStep = None
for i in (tq := tqdm(range(nSteps), leave = False)):
    begin = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    begin.record()
    result = integrator.function(
        state = runningState,
        f = fn,
        dt = config.dt,  
        config = config,
        schemeConfig = schemeConfig,
        verbose = False,
        # priorStep = priorStep
    )
    kes.append(torch.sum(0.5 * result.state.state.masses * torch.sum(result.state.state.velocities**2, dim=1)))
    # print('max_vel:', torch.linalg.norm(result.state.state.velocities, dim = -1).max())
    end.record()
    torch.cuda.synchronize()
    priorStep = result.stages[-1]
    timing = begin.elapsed_time(end)

    runningState = result.state
    t = runningState.t
    # schemeConfig.rigidBodies[0].linearVelocity = 0.5 * torch.cos(t * np.pi * 2)
    # linearVelocity = 0.5 * torch.cos(t * np.pi * 2)

    currentState = runningState.state
    # print(f'-' * 80)
    # print(f'Fluid density stats: min={currentState.densities[currentState.kinds == 0].min().item()}, max={currentState.densities[currentState.kinds == 0].max().item()}, mean={currentState.densities[currentState.kinds == 0].mean().item()}')
    # print(f'Boundary density stats: min={currentState.densities[currentState.kinds == 1].min().item()}, max={currentState.densities[currentState.kinds == 1].max().item()}, mean={currentState.densities[currentState.kinds == 1].mean().item()}')
    if i % 20 == 0 :
        # densities = computeDensities(runningState.state, config, schemeConfig, None)

        plotter.updateQuantities(
            {
                "A": runningState.state.velocities,
                # "B": runningState.state.densities,
                "B": runningState.state.UIDs
            },
            newParticleState = runningState.state,
        )
        # plotter.export(f'{imagePath}/frame_{i:05d}.png', dpi = 300)
    # break
        
    maxVel = torch.linalg.norm(runningState.state.velocities, dim = -1).max()
    tq.set_description(f"Step {i+1}/{nSteps}, time: {(i+1)*config.dt:8.4g}/{t_limit:8.4g} | max vel: {maxVel:.3g} | iter time: {timing:.3f} ms")
    # t = {runningState.t:2f}, dt = {config.dt:.3g}, ptcls = {len(runningState.state.positions)}\nTotal Energy: {totalEnergy:.3g}, Kinetic Energy: {kineticEnergy:.3g}, Thermal Energy: {thermalEnergy:.3g}'
    # break
    if torch.any(torch.isnan(runningState.state.velocities)):
        print("NaN detected in velocities, stopping simulation.")
        break



  0%|          | 0/6400 [00:00<?, ?it/s]

In [15]:
config.kernel

<KernelFunctions.Wendland4: 1>

In [ ]:
print(trajectory.attrs.keys())

In [ ]:
caseName = trajectory.attrs['caseName']
timeLimit = trajectory.attrs['timeLimit']
# dt = trajectory['states']['frame_00001'].attrs['time'] - trajectory['states']['frame_00000'].attrs['time']
nx = trajectory.attrs['nx']
n_h = trajectory.attrs['n_h']
L = trajectory.attrs['L']
W = trajectory.attrs['W']
obstacleType = trajectory.attrs['obstacleType']
aoa = trajectory.attrs['aoa']
obstacleActive = trajectory.attrs['obstacleActive']

dt = loadedConfig['config']['dt']
fixedSoundSpeed = loadedConfig['schemeConfig']['fixedSoundSpeed']

device = torch.device('cpu')
dtype = torch.float32

# dx = loadedConfig['config']['dx']
# band = trajectory.attrs['band']
dim = loadedConfig['config']['dim']
domain = buildDomainDescription(L, dim, True, device, dtype)
domain.min = torch.tensor(loadedConfig['config']['domain']['min'], device=device, dtype=dtype)
domain.max = torch.tensor(loadedConfig['config']['domain']['max'], device=device, dtype=dtype)

In [ ]:
markerSize = 6
plotWidth = 28
plotHeight = 10

In [ ]:
stateIndex = 0
state, time = load_state(stateIndex, trajectory)

caseText = f'{caseName}'
timeText = f't = {time:.4g}/{timeLimit:.4g} | dt = {dt:.4g}'
particleText = f'particles = {len(state.positions[state.kinds == 0])} fluid + {len(state.positions[state.kinds == 1])} boundary | nx = {nx} | n_h = {n_h}'
domainText = f'L = {L}, W = {W}'
obstacleText = f'obstacle: {obstacleType}, aoa: {aoa}' if obstacleActive else 'no obstacle'
stateText = f'v_max = {state.velocities.max().cpu().item():.4g} (c0 = {fixedSoundSpeed:.4g}), rho_max = {state.densities.max().cpu().item():.4g}, rho_min = {state.densities.min().cpu().item():.4g}'
timingText = f'iter time: {0.00:.3f} ms'

titleString = f'{caseText} | {timeText} | {particleText} | {domainText} | {obstacleText} | {stateText} | {timingText}'

from ipywidgets import widgets

def update_frame(frame_index):
    global state, time
    state, time = load_state(frame_index, trajectory)
    plotter.updateQuantities({
        # "A": state.velocities,
        "A": state.UIDs
    }, newParticleState=state)
    caseText = f'{caseName}'
    timeText = f't = {time:.4g}/{timeLimit:.4g} | dt = {dt:.4g}'
    particleText = f'particles = {len(state.positions[state.kinds == 0])} fluid + {len(state.positions[state.kinds == 1])} boundary | nx = {nx} | n_h = {n_h}'
    domainText = f'L = {L}, W = {W}'
    obstacleText = f'obstacle: {obstacleType}, aoa: {aoa}' if obstacleActive else 'no obstacle'
    stateText = f'v_max = {state.velocities.max().cpu().item():.4g} (c0 = {fixedSoundSpeed:.4g}), rho_max = {state.densities.max().cpu().item():.4g}, rho_min = {state.densities.min().cpu().item():.4g}'
    timingText = f'iter time: {0.00:.3f} ms'

    titleString = f'{caseText} | {timeText} | {particleText} | {domainText} | {obstacleText} | {stateText} | {timingText}'

    plotter.updateTitle(titleString)


markerSize = 12
velocityPlot = PlottingOptions(
            colorMap = UniformColorMap.viridis,
            markerSize = markerSize,
            midPoint = 0.0,
            quantityScaling = PlotScaling.Linear,
            mapping = Mapping.L2Norm,
            plotTitle = "Particle Velocity Magnitude",
            plotTitleGap = 0.08,
            boundaryVisualization = VisualizeOptions.Visualize,
            # gridVisualization = GridVisualization(
            #     resolution = 1024,
            #     streamLines = True,
            # ),

            # vMin=1e-10,
            vMin = 0.0,
            vMax = fixedSoundSpeed * 0.1,
        )
densityPlot = PlottingOptions(
            colorMap = DivergingColorMap.RdBu,
            flipColorMap=True,
            markerSize = markerSize,
            midPoint = 1.0,
            quantityScaling = PlotScaling.Linear,
            plotTitle = "Particle Density",
            # vMin = 0.95,
            # vMax = 1.05,
            plotTitleGap = 0.08,
            # gridVisualization = GridVisualization(
            #     resolution = 512,
            # ),
        )
UIDPlot = PlottingOptions(
            colorMap = CyclicColorMap.twilight,
            # flipColorMap=True,
            markerSize = markerSize,
            # midPoint = 1.0,
            quantityScaling = PlotScaling.Linear,
            plotTitle = f"Particle IDs ({caseName},  {len(state.positions[state.kinds == 0])} fluid + {len(state.positions[state.kinds == 1])} boundary particles)",
            boundaryVisualization = VisualizeOptions.Passive,
            # vMin = 0.95,
            # vMax = 1.05
            plotTitleGap = 0.08,
            # gridVisualization = GridVisualization(
            #     resolution = 512,
            # ),
        )


In [ ]:

plotter = visualize(
    particleState = state,
    domain = domain,
    quantities = {
        # "A": state.velocities,
        "A": state.UIDs
    },
    plotOptions = {
        # "A": velocityPlot,
        "A": UIDPlot
    },
    figTitle = titleString,
    mosaic = 'A',
    figsize= (plotWidth, plotHeight),
    backend='vispy',
    # backend='pyVista',
    # backendOptions = {
    #     # In notebooks, use trame for reliable live updates.
    #     'jupyter_backend': 'trame',
    # }
)

plotter.updateTitle(titleString)


frame_slider = widgets.IntSlider(
    value=500,
    min=0,
    max=numStates-1,
    step=1,
    description='Frame:',
    continuous_update=True,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='80%')
)

frame_slider.observe(lambda change: update_frame(change['new']), names='value')
# update_frame(frame_slider.value)

export_button = widgets.Button(
    description='Export Current Frame',
    button_style='success',
    tooltip='Export the current frame as an image',
    icon='download'
)
export_button.on_click(lambda b: plotter.export(f'{caseName}_frame_{frame_slider.value:05d}.png'))


display(frame_slider)
display(export_button)


In [ ]:
uMax = torch.linalg.norm(state.velocities, dim=1).max().cpu().item()
dx = state.masses[state.kinds == 0].mean().cpu().item() ** (1.0 / dim)  # approximate particle spacing based on mass and density
cfl = uMax * dt / dx
print(f"Max velocity: {uMax:.4g}, dx: {dx:.4g}, dt: {dt:.4g}, CFL number: {cfl:.4g}")

targetCFL = 1.0
maxDt = targetCFL * dx / uMax
print(f"Target CFL: {targetCFL:.4g}, Max dt for target CFL: {maxDt:.4g}")
dtRatio = dt / maxDt
print(f"dt ratio: {dtRatio:.4g} (dt / maxDt for target CFL)")
print(f'Maximum Coarse Graining Factor: {1/dtRatio:.4g} (1 / dtRatio for target CFL)')